# Task substitution in Qwen3.5-4B

**Hypothesis.** When a prompt looks like a familiar task but differs in one key detail, a language model often solves the familiar, easier task instead. The result looks like a hallucination, but it is actually a correct answer to a different question.

**How we test it.** Every item is a *minimal pair*: a canonical (familiar) prompt and a perturbed prompt that changes one detail. For each perturbed item we store two answers:

- `gold`: the correct answer to the task as asked
- `shortcut`: the answer to the familiar version

If wrong answers land on `shortcut` far more often than on anything else, the errors are **substitution**, not random fabrication. Each pair is asked under four conditions:

| condition | prompt |
|---|---|
| `canonical` | the familiar task (reference) |
| `perturbed` | the changed task, no warning |
| `nudge` | the changed task plus a generic "read carefully" warning |
| `explicit` | the changed task, with the changed detail pointed out |

A large gain from `perturbed` to `nudge`/`explicit` means the model *could* do the task but took the shortcut by default.

**Runtime.** Use a GPU runtime. On an L4 or A100, use `ENGINE = "vllm"`: the full run (both modes) takes well under an hour. On a free T4, use `ENGINE = "hf"`. Thinking mode is slow there, so set `PAIRS_PER_CATEGORY["think"]` to about 10.

**Run order.** Run top to bottom. Results are saved to your Hugging Face Bucket after every batch. If Colab disconnects, re-run from the top: generation resumes where it stopped.

## 0 · Configuration

In [ ]:
MODEL_ID = "Qwen/Qwen3.5-4B"
ENGINE = "vllm"                  # "vllm" (fast, L4/A100) or "hf" (plain transformers, works on a T4)
MODES = ["no_think", "think"]    # Qwen3.5 thinks by default; we evaluate both
MAX_NEW_TOKENS = {"no_think": 1024, "think": 8192}
PAIRS_PER_CATEGORY = {"no_think": None, "think": None}   # None = all pairs; e.g. 2 for a smoke test
N_SAMPLES = 1                    # completions per prompt (3 gives per-item rates if time allows)
SEED = 0
HF_BUCKET = "YOUR_HF_USERNAME/llm-task-shortcutting"     # set to None to keep results only on the runtime
HF_BATCH_SIZE = 16               # only used when ENGINE == "hf"

# Qwen's recommended sampling, with presence_penalty left at 0 so code and verbatim copying are not
# penalised, and so the "hf" engine (which has no presence penalty) behaves the same as vLLM.
SAMPLING = {
    "no_think": dict(temperature=0.7, top_p=0.8, top_k=20),
    "think": dict(temperature=0.6, top_p=0.95, top_k=20),
}

In [ ]:
import subprocess, sys

packages = ["transformers", "accelerate", "huggingface_hub"]   # pandas / matplotlib: keep Colab's versions
if ENGINE == "vllm":
    packages.append("vllm")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *packages], check=True)
# If this upgraded torch (vLLM sometimes does), restart the session once and run from the top again.

### Storage

Colab wipes `/content` when the runtime resets, so results are synced to a Hugging Face Bucket. Create a token with **write** access at https://huggingface.co/settings/tokens. Store it as a Colab secret named `HF_TOKEN`, or paste it when asked. The bucket stays private until the last cell publishes it.

In [ ]:
import json
from pathlib import Path

OUT = Path("/content/artifacts")
for sub in ("data", "results", "figures"):
    (OUT / sub).mkdir(parents=True, exist_ok=True)
DATA_PATH = OUT / "data" / "shortcut_pairs.jsonl"

if HF_BUCKET:
    assert "YOUR_HF_USERNAME" not in HF_BUCKET, "Set HF_BUCKET in the configuration cell first."
    from huggingface_hub import login, create_bucket, sync_bucket
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None
    if not token:
        from getpass import getpass
        token = getpass("Hugging Face token with write access: ")
    login(token=token)
    create_bucket(HF_BUCKET, private=True, exist_ok=True)
    sync_bucket(f"hf://buckets/{HF_BUCKET}", str(OUT), quiet=True)   # pull earlier results to resume
    print("bucket:", f"https://huggingface.co/buckets/{HF_BUCKET}")


def sync_up():
    if HF_BUCKET:
        sync_bucket(str(OUT), f"hf://buckets/{HF_BUCKET}", quiet=True)

## 1 · Dataset

### 1.1 Source items
Hand-written material. Each entry is a familiar task plus one changed detail; `hint` names that detail and is used only in the `explicit` condition. To add items, append to these lists and re-run the section.

In [ ]:
# Modified classic puzzles. checker "number": gold values are acceptable numbers.
# checker "text": gold values are acceptable words (whole word, case-insensitive).
PUZZLES = [
    dict(id="bat_ball", checker="number",
         canonical="A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. "
                   "How much does the ball cost, in cents?",
         perturbed="A bat and a ball cost $1.10 in total. The bat costs $1.00. "
                   "How much does the ball cost, in cents?",
         hint="The bat costs exactly $1.00, not $1.00 more than the ball.",
         gold_canonical=[5, 0.05], gold_perturbed=[10, 0.10]),
    dict(id="widgets", checker="number",
         canonical="If it takes 5 machines 5 minutes to make 5 widgets, how many minutes would it take "
                   "100 machines to make 100 widgets?",
         perturbed="If it takes 5 machines 5 minutes to make 5 widgets, how many minutes would it take "
                   "5 machines to make 100 widgets?",
         hint="The number of machines stays at 5.",
         gold_canonical=[5], gold_perturbed=[100]),
    dict(id="lily_pads", checker="number",
         canonical="A patch of lily pads doubles in size every day. It takes 48 days for the patch to "
                   "cover the whole lake. How many days does it take to cover half of the lake?",
         perturbed="A patch of lily pads grows by the same fixed area every day, starting from nothing. "
                   "It takes 48 days for the patch to cover the whole lake. How many days does it take "
                   "to cover half of the lake?",
         hint="The growth is linear (the same amount every day), not doubling.",
         gold_canonical=[47], gold_perturbed=[24]),
    dict(id="river_crossing", checker="number",
         canonical="A farmer must cross a river with a wolf, a goat and a cabbage. The boat can carry the "
                   "farmer and at most one item. The wolf cannot be left alone with the goat, and the goat "
                   "cannot be left alone with the cabbage. What is the minimum number of one-way boat "
                   "crossings needed to get everything across?",
         perturbed="A farmer must cross a river with a wolf, a goat and a cabbage. The boat is big enough "
                   "to carry the farmer and all three items at once. What is the minimum number of one-way "
                   "boat crossings needed to get everything across?",
         hint="The boat can carry everything in a single trip.",
         gold_canonical=[7], gold_perturbed=[1]),
    dict(id="surgeon", checker="text",
         canonical="A father and his son are in a car accident. The father dies at the scene. The son is "
                   "rushed to hospital, where the surgeon says, \"I can't operate on this boy, he's my son!\" "
                   "Who is the surgeon to the boy? Answer in one word.",
         perturbed="A boy and his mother are in a car accident and are both taken to hospital. The surgeon, "
                   "who is the boy's father, says, \"I can't operate on this boy, he's my son!\" Who is the "
                   "surgeon to the boy? Answer in one word.",
         hint="The question states directly who the surgeon is.",
         gold_canonical=["mother", "mom", "mum"], gold_perturbed=["father", "dad"]),
    dict(id="feathers", checker="text",
         canonical="Which weighs more, a pound of feathers or a pound of bricks?",
         perturbed="Which weighs more, a pound of feathers or two pounds of bricks?",
         hint="The amounts are different: one pound versus two pounds.",
         gold_canonical=["same", "equal", "equally", "neither"], gold_perturbed=["bricks", "brick"]),
    dict(id="ladder_tide", checker="number",
         canonical="A rope ladder hangs over the side of a ship floating in a harbour. Its rungs are 1 foot "
                   "apart and 3 rungs are underwater. The tide rises 1 foot per hour. After 3 hours, how "
                   "many rungs are underwater?",
         perturbed="A long rope ladder hangs from a fixed concrete pier that does not move. Its rungs are "
                   "1 foot apart and 3 rungs are underwater. The water level rises 1 foot per hour. After "
                   "3 hours, how many rungs are underwater?",
         hint="The pier is fixed, so the ladder does not rise with the water.",
         gold_canonical=[3], gold_perturbed=[6]),
    dict(id="socks", checker="number",
         canonical="A drawer contains 10 black socks and 10 white socks. In the dark, what is the minimum "
                   "number of socks you must take out to be sure of having a matching pair?",
         perturbed="A drawer contains 10 black socks and 10 white socks. In the dark, what is the minimum "
                   "number of socks you must take out to be sure of having two socks of different colours?",
         hint="You need two socks of different colours, not a matching pair.",
         gold_canonical=[3], gold_perturbed=[11]),
    dict(id="months_28", checker="number",
         canonical="How many months of the year have 28 days?",
         perturbed="In a non-leap year, how many months have exactly 28 days and no more?",
         hint="Count only months whose total length is exactly 28 days.",
         gold_canonical=[12], gold_perturbed=[1]),
    dict(id="race_place", checker="number",
         canonical="In a race, you overtake the person in second place. What place are you in now?",
         perturbed="In a race, you overtake the person in third place. What place are you in now?",
         hint="The person you overtake was in third place.",
         gold_canonical=[2], gold_perturbed=[3]),
    dict(id="sister_age", checker="number",
         canonical="When I was 6, my sister was half my age. Now I am 70. How old is my sister?",
         perturbed="When I was 6, my sister was twice my age. Now I am 70. How old is my sister?",
         hint="The sister was twice my age, so she is the older one.",
         gold_canonical=[67], gold_perturbed=[76]),
    dict(id="fifth_daughter", checker="text",
         canonical="Mary's father has five daughters: Nana, Nene, Nini and Nono. What is the name of the "
                   "fifth daughter? Answer with just the name.",
         perturbed="Mary's uncle has five daughters: Nana, Nene, Nini, Nono and Nunu. What is the name of "
                   "his fifth daughter? Answer with just the name.",
         hint="All five daughters are listed, and Mary is his niece, not his daughter.",
         gold_canonical=["Mary"], gold_perturbed=["Nunu"]),
    dict(id="cats_mice", checker="number",
         canonical="If 3 cats catch 3 mice in 3 minutes, how many cats are needed to catch 100 mice in "
                   "100 minutes?",
         perturbed="If 3 cats catch 3 mice in 3 minutes, how many cats are needed to catch 100 mice in "
                   "3 minutes?",
         hint="The time allowed stays at 3 minutes.",
         gold_canonical=[3], gold_perturbed=[100]),
    dict(id="monty_transparent", checker="number",
         canonical="On a game show there are three doors: behind one is a car, behind the other two are "
                   "goats. You pick door 1. The host, who knows where the car is, opens door 3 to reveal a "
                   "goat and offers you the chance to switch to door 2. What is the probability of winning "
                   "the car if you switch? Give a fraction.",
         perturbed="On a game show there are three transparent doors: behind one is a car, behind the other "
                   "two are goats. You pick door 1 because you can see the car behind it. The host opens "
                   "door 3 to reveal a goat and offers you the chance to switch to door 2. What is the "
                   "probability of winning the car if you switch? Give a fraction.",
         hint="The doors are transparent, so you already know the car is behind door 1.",
         gold_canonical=[2 / 3], gold_perturbed=[0]),
    dict(id="monty_ignorant_host", checker="number",
         canonical="On a game show there are three doors: behind one is a car, behind the other two are "
                   "goats. You pick door 1. The host, who knows where the car is, opens door 3 to reveal a "
                   "goat and offers you the chance to switch to door 2. What is the probability of winning "
                   "the car if you switch? Give a fraction.",
         perturbed="On a game show there are three doors: behind one is a car, behind the other two are "
                   "goats. You pick door 1. The host does not know where the car is; he opens door 3 at "
                   "random and it happens to reveal a goat. He offers you the chance to switch to door 2. "
                   "What is the probability of winning the car if you switch? Give a fraction.",
         hint="The host opened a door at random, without knowing where the car is.",
         gold_canonical=[2 / 3], gold_perturbed=[1 / 2]),
    dict(id="sally_sisters", checker="number",
         canonical="Sally has 3 brothers. Each brother has 2 sisters. How many sisters does Sally have?",
         perturbed="Sally has 3 brothers. Each brother has 3 sisters. How many sisters does Sally have?",
         hint="Each brother has 3 sisters, and Sally is one of them.",
         gold_canonical=[1], gold_perturbed=[2]),
    dict(id="snail_well", checker="number",
         canonical="A snail is at the bottom of a 10-metre well. Each day it climbs up 3 metres and each "
                   "night it slides back down 2 metres. On which day does it reach the top?",
         perturbed="A snail is at the bottom of a 10-metre well. Each day it climbs up 3 metres, and at "
                   "night it rests without sliding back at all. On which day does it reach the top?",
         hint="The snail does not slide back at night.",
         gold_canonical=[8], gold_perturbed=[4]),
]

# Letter counting on familiar words. The perturbation adds or removes one occurrence of the
# target letter, so the count for the familiar spelling is the shortcut.
COUNT_WORDS = [
    ("strawberry", "r"), ("banana", "a"), ("mississippi", "s"), ("committee", "t"),
    ("necessary", "s"), ("bookkeeper", "e"), ("assessment", "s"), ("occurrence", "c"),
    ("raspberry", "r"), ("parallel", "l"), ("embarrass", "r"), ("cappuccino", "p"),
    ("tomorrow", "r"), ("possession", "s"), ("balloon", "l"), ("coffee", "f"),
    ("address", "d"), ("success", "c"), ("millennium", "n"), ("accommodate", "m"),
]

# Famous sentences, each with one meaning-level word swap. A spelling typo is also generated
# automatically for each sentence.
PROVERBS = [
    ("An apple a day keeps the doctor away.", "doctor", "dentist"),
    ("The early bird catches the worm.", "worm", "train"),
    ("Actions speak louder than words.", "words", "music"),
    ("Don't count your chickens before they hatch.", "chickens", "ducks"),
    ("A picture is worth a thousand words.", "thousand", "hundred"),
    ("The pen is mightier than the sword.", "sword", "spoon"),
    ("When in Rome, do as the Romans do.", "Romans", "Greeks"),
    ("All that glitters is not gold.", "gold", "silver"),
    ("Curiosity killed the cat.", "cat", "dog"),
    ("Every cloud has a silver lining.", "silver", "golden"),
    ("Where there's smoke, there's fire.", "fire", "steam"),
    ("The quick brown fox jumps over the lazy dog.", "dog", "cat"),
    ("To be or not to be, that is the question.", "question", "answer"),
    ("Rome wasn't built in a day.", "day", "week"),
    ("Two heads are better than one.", "one", "three"),
    ("A journey of a thousand miles begins with a single step.", "step", "stone"),
    ("You can't judge a book by its cover.", "cover", "title"),
    ("Practice makes perfect.", "perfect", "progress"),
    ("Look before you leap.", "leap", "sleep"),
    ("Birds of a feather flock together.", "together", "alone"),
]

# "List N items" with an unusual N; the familiar defaults (5, 10) are the shortcut.
LIST_TOPICS = [
    "fruits", "countries in Europe", "programming languages", "musical instruments",
    "ocean animals", "sports", "vegetables", "colours", "famous scientists", "board games",
    "types of trees", "kitchen utensils", "car brands", "rivers", "birds", "flowers",
    "languages", "dog breeds", "cheeses", "capital cities",
]

# Textbook functions with a changed spec. The reference solutions are run to compute the expected
# outputs, and the self-test feeds them back through the grader.
CODE_TASKS = [
    dict(id="sort_even_first", func="sort_numbers", sig="xs",
         canonical="takes a list of integers and returns a new list sorted in ascending order",
         perturbed="takes a list of integers and returns a new list in which all even numbers come before "
                   "all odd numbers, with each group sorted in ascending order",
         hint="All even numbers must come first, before any odd number.",
         ref_canonical="def sort_numbers(xs):\n    return sorted(xs)\n",
         ref_perturbed="def sort_numbers(xs):\n"
                       "    return sorted(x for x in xs if x % 2 == 0) + sorted(x for x in xs if x % 2)\n",
         tests=[[[5, 2, 9, 4, 1]], [[3, 1, 2]], [[10, 7, 8, 6, 1]], [[1]], [[]]]),
    dict(id="fizzbuzz_4_7", func="fizzbuzz", sig="n",
         canonical="returns a list of strings for the numbers 1 to n: 'Fizz' for multiples of 3, 'Buzz' for "
                   "multiples of 5, 'FizzBuzz' for multiples of both, and otherwise the number itself as a string",
         perturbed="returns a list of strings for the numbers 1 to n: 'Fizz' for multiples of 4, 'Buzz' for "
                   "multiples of 7, 'FizzBuzz' for multiples of both, and otherwise the number itself as a string",
         hint="The divisors are 4 and 7, not 3 and 5.",
         ref_canonical="def fizzbuzz(n):\n    out = []\n    for i in range(1, n + 1):\n"
                       "        s = ('Fizz' if i % 3 == 0 else '') + ('Buzz' if i % 5 == 0 else '')\n"
                       "        out.append(s or str(i))\n    return out\n",
         ref_perturbed="def fizzbuzz(n):\n    out = []\n    for i in range(1, n + 1):\n"
                       "        s = ('Fizz' if i % 4 == 0 else '') + ('Buzz' if i % 7 == 0 else '')\n"
                       "        out.append(s or str(i))\n    return out\n",
         tests=[[15], [28], [30]]),
    dict(id="palindrome_strict", func="is_palindrome", sig="s",
         canonical="returns True if the string s reads the same forwards and backwards, ignoring case, spaces "
                   "and punctuation, and False otherwise",
         perturbed="returns True if the string s reads exactly the same forwards and backwards, character for "
                   "character (case-sensitive, with spaces and punctuation counted), and False otherwise",
         hint="The comparison is strict: case, spaces and punctuation all count.",
         ref_canonical="def is_palindrome(s):\n    t = [c.lower() for c in s if c.isalnum()]\n"
                       "    return t == t[::-1]\n",
         ref_perturbed="def is_palindrome(s):\n    return s == s[::-1]\n",
         tests=[["Racecar"], ["A man, a plan, a canal: Panama"], ["abba"], ["abc"], ["No lemon, no melon"]]),
    dict(id="fib_lucas", func="fib", sig="n",
         canonical="returns the n-th term of the Fibonacci sequence, where fib(0) = 0, fib(1) = 1 and every "
                   "later term is the sum of the two before it",
         perturbed="returns the n-th term of the sequence where fib(0) = 2, fib(1) = 1 and every later term "
                   "is the sum of the two before it",
         hint="The sequence starts with 2, 1, not 0, 1.",
         ref_canonical="def fib(n):\n    a, b = 0, 1\n    for _ in range(n):\n        a, b = b, a + b\n"
                       "    return a\n",
         ref_perturbed="def fib(n):\n    a, b = 2, 1\n    for _ in range(n):\n        a, b = b, a + b\n"
                       "    return a\n",
         tests=[[0], [1], [5], [10]]),
    dict(id="reverse_letters", func="reverse_words", sig="s",
         canonical="takes a string of words separated by single spaces and returns the string with the order "
                   "of the words reversed",
         perturbed="takes a string of words separated by single spaces and returns the string with the letters "
                   "of each word reversed, keeping the words in their original order",
         hint="Reverse the letters inside each word; the word order stays the same.",
         ref_canonical="def reverse_words(s):\n    return ' '.join(s.split(' ')[::-1])\n",
         ref_perturbed="def reverse_words(s):\n    return ' '.join(w[::-1] for w in s.split(' '))\n",
         tests=[["hello world"], ["the quick brown fox"], ["a"]]),
    dict(id="vowels_with_y", func="count_vowels", sig="s",
         canonical="returns the number of vowels (a, e, i, o, u) in the string s, case-insensitive",
         perturbed="returns the number of vowels in the string s, case-insensitive, where the vowels are "
                   "a, e, i, o, u and also y",
         hint="The letter y counts as a vowel.",
         ref_canonical="def count_vowels(s):\n    return sum(c in 'aeiou' for c in s.lower())\n",
         ref_perturbed="def count_vowels(s):\n    return sum(c in 'aeiouy' for c in s.lower())\n",
         tests=[["rhythm"], ["Happy day"], ["AEIOU"], ["sky"]]),
    dict(id="double_factorial", func="fact", sig="n",
         canonical="returns n! (the product 1 * 2 * ... * n), with fact(0) == 1",
         perturbed="returns the double factorial n!! (the product n * (n-2) * (n-4) * ... down to 1 or 2), "
                   "with fact(0) == 1",
         hint="This is the double factorial: it multiplies every second number.",
         ref_canonical="def fact(n):\n    r = 1\n    for i in range(2, n + 1):\n        r *= i\n    return r\n",
         ref_perturbed="def fact(n):\n    r = 1\n    while n > 1:\n        r *= n\n        n -= 2\n    return r\n",
         tests=[[0], [1], [5], [6], [8]]),
    dict(id="julian_leap", func="is_leap", sig="year",
         canonical="returns True if the year is a leap year in the Gregorian calendar, and False otherwise",
         perturbed="returns True if the year is a leap year in the Julian calendar, where every year divisible "
                   "by 4 is a leap year with no other exceptions, and False otherwise",
         hint="Julian rule: divisible by 4 means leap year, with no century exceptions.",
         ref_canonical="def is_leap(year):\n    return year % 4 == 0 and (year % 100 != 0 or year % 400 == 0)\n",
         ref_perturbed="def is_leap(year):\n    return year % 4 == 0\n",
         tests=[[1900], [2000], [2024], [2023], [2100]]),
    dict(id="caesar_back", func="caesar", sig="s",
         canonical="shifts every lowercase letter in s 3 positions forward in the alphabet (wrapping from z "
                   "back to c) and leaves all other characters unchanged",
         perturbed="shifts every lowercase letter in s 3 positions backward in the alphabet (wrapping from a "
                   "to x) and leaves all other characters unchanged",
         hint="The shift is backward, not forward.",
         ref_canonical="def caesar(s):\n"
                       "    return ''.join(chr((ord(c) - 97 + 3) % 26 + 97) if 'a' <= c <= 'z' else c for c in s)\n",
         ref_perturbed="def caesar(s):\n"
                       "    return ''.join(chr((ord(c) - 97 - 3) % 26 + 97) if 'a' <= c <= 'z' else c for c in s)\n",
         tests=[["abc"], ["xyz"], ["hello, world"]]),
    dict(id="capitalize_last", func="title_case", sig="s",
         canonical="takes a string of words separated by single spaces and returns it with the first letter of "
                   "each word uppercase and all other letters lowercase",
         perturbed="takes a string of words separated by single spaces and returns it with the last letter of "
                   "each word uppercase and all other letters lowercase",
         hint="Capitalize the last letter of each word, not the first.",
         ref_canonical="def title_case(s):\n"
                       "    return ' '.join(w[:1].upper() + w[1:].lower() for w in s.split(' '))\n",
         ref_perturbed="def title_case(s):\n"
                       "    return ' '.join(w[:-1].lower() + w[-1:].upper() for w in s.split(' '))\n",
         tests=[["hello world"], ["the quick brown fox"], ["python"]]),
]

### 1.2 Build the minimal pairs
Arithmetic operands, misspellings and typos are generated from `SEED`, so the dataset rebuilds exactly.

In [ ]:
import random
import re
from collections import Counter

ANSWER_FORMAT = "\n\nEnd your response with a final line of the form 'Answer: <your answer>'."
NUDGE = ("Read the question carefully. It may differ in an important way from a similar, "
         "well-known question.\n\n")
CONDITIONS = ["canonical", "perturbed", "nudge", "explicit"]


def emit_pair(rows, pair_id, category, *, canonical, perturbed, hint, gold_canonical,
              gold_perturbed, checker, suffix="", meta=None):
    """Append one minimal pair under all four conditions. Non-canonical rows carry the canonical
    answer as `shortcut`."""
    variants = {
        "canonical": (canonical + suffix, gold_canonical, None),
        "perturbed": (perturbed + suffix, gold_perturbed, gold_canonical),
        "nudge": (NUDGE + perturbed + suffix, gold_perturbed, gold_canonical),
        "explicit": (f"{perturbed}\n\nNote: {hint}{suffix}", gold_perturbed, gold_canonical),
    }
    for condition, (prompt, gold, shortcut) in variants.items():
        rows.append({"id": f"{pair_id}-{condition}", "pair_id": pair_id, "category": category,
                     "condition": condition, "prompt": prompt, "gold": gold, "shortcut": shortcut,
                     "checker": checker, "meta": meta or {}})


def gen_puzzles(rows, rng):
    for p in PUZZLES:
        emit_pair(rows, f"puzzle-{p['id']}", "modified_puzzle",
                  canonical=p["canonical"], perturbed=p["perturbed"], hint=p["hint"],
                  gold_canonical=p["gold_canonical"], gold_perturbed=p["gold_perturbed"],
                  checker={"type": p["checker"]}, suffix=ANSWER_FORMAT)


def to_base(n, base):
    digits = ""
    while n:
        digits = str(n % base) + digits
        n //= base
    return digits or "0"


def gen_arithmetic(rows, rng, n_pairs=40):
    seen = set()
    while len(seen) < n_pairs:
        base, n_digits = rng.choice([8, 9]), rng.choice([2, 3])
        a, b = (str(rng.randint(1, base - 1)) + "".join(str(rng.randint(0, base - 1)) for _ in range(n_digits - 1))
                for _ in range(2))
        decimal_sum = int(a) + int(b)
        base_sum = to_base(int(a, base) + int(b, base), base)
        # Without a carry both readings agree and the pair would not discriminate.
        if str(decimal_sum) == base_sum or (base, a, b) in seen:
            continue
        seen.add((base, a, b))
        emit_pair(rows, f"arith-{len(seen):03d}", "counterfactual_arithmetic",
                  canonical=f"What is {a} + {b}?",
                  perturbed=f"In base {base}, what is {a} + {b}? All numbers are written in base {base}.",
                  hint=f"In base {base} the digits run from 0 to {base - 1}, so a column carries when its "
                       f"sum reaches {base}, not 10.",
                  gold_canonical=[decimal_sum], gold_perturbed=[int(base_sum)],
                  checker={"type": "number"}, suffix=ANSWER_FORMAT,
                  meta={"base": base, "a": a, "b": b})


def gen_counting(rows, rng):
    k = 0
    question = "How many times does the letter '{}' appear in \"{}\"?"
    for word, letter in COUNT_WORDS:
        positions = [i for i, c in enumerate(word) if c == letter]
        edits = [("insert", rng.choice(positions))]
        if len(positions) >= 2:
            edits.append(("delete", rng.choice(positions)))
        for kind, pos in edits:
            misspelled = word[:pos] + letter + word[pos:] if kind == "insert" else word[:pos] + word[pos + 1:]
            k += 1
            emit_pair(rows, f"count-{k:03d}", "misspelled_counting",
                      canonical=question.format(letter, word),
                      perturbed=question.format(letter, misspelled),
                      hint="The word is deliberately misspelled. Count the letters exactly as written.",
                      gold_canonical=[word.count(letter)], gold_perturbed=[misspelled.count(letter)],
                      checker={"type": "number"}, suffix=ANSWER_FORMAT,
                      meta={"word": word, "misspelled": misspelled, "letter": letter, "edit": kind})


COPY_PROMPT = ("Repeat the following sentence exactly as written, character for character. "
               "Output only the sentence.\n\n{}")
COPY_HINT = ("The sentence intentionally differs from the version you may know. Keep it exactly as "
             "written and do not correct anything.")


def make_typo(rng, sentence):
    """Swap two adjacent interior letters of a word that appears exactly once."""
    words = re.findall(r"[A-Za-z]+", sentence)
    counts = Counter(words)
    candidates = [w for w in words if len(w) >= 5 and counts[w] == 1]
    rng.shuffle(candidates)
    for w in candidates:
        spots = [i for i in range(1, len(w) - 2) if w[i] != w[i + 1]]
        if spots:
            i = rng.choice(spots)
            return w, w[:i] + w[i + 1] + w[i] + w[i + 2:]
    raise ValueError(f"no typo candidate in {sentence!r}")


def replace_word(sentence, old, new):
    return re.sub(rf"(?<!\w){re.escape(old)}(?!\w)", new, sentence, count=1)


def gen_copy(rows, rng):
    k = 0
    for sentence, old, new in PROVERBS:
        typo_src, typo = make_typo(rng, sentence)
        for kind, src_word, new_word in (("word_swap", old, new), ("typo", typo_src, typo)):
            k += 1
            changed = replace_word(sentence, src_word, new_word)
            emit_pair(rows, f"copy-{k:03d}", "verbatim_copy",
                      canonical=COPY_PROMPT.format(sentence), perturbed=COPY_PROMPT.format(changed),
                      hint=COPY_HINT,
                      gold_canonical={"text": sentence, "token": src_word},
                      gold_perturbed={"text": changed, "token": new_word},
                      checker={"type": "exact_copy"}, meta={"edit": kind})


def gen_lists(rows, rng):
    template = "List {} {}. Write one item per line, numbered."
    for k, topic in enumerate(LIST_TOPICS, 1):
        n = rng.choice([7, 8, 9, 11, 12, 13])
        emit_pair(rows, f"list-{k:03d}", "list_length",
                  canonical=template.format(10, topic), perturbed=template.format(n, topic),
                  hint=f"The list must contain exactly {n} items, no more and no fewer.",
                  gold_canonical=10, gold_perturbed=n,
                  checker={"type": "list_count", "shortcut_counts": [5, 10]}, meta={"topic": topic})


def run_reference(src, func, tests):
    namespace = {}
    exec(src, namespace)  # trusted: our own reference solutions
    return json.loads(json.dumps([namespace[func](*args) for args in tests]))


def gen_code(rows, rng):
    template = "Write a Python function `{}({})` that {}."
    for t in CODE_TASKS:
        expected_c = run_reference(t["ref_canonical"], t["func"], t["tests"])
        expected_p = run_reference(t["ref_perturbed"], t["func"], t["tests"])
        assert expected_c != expected_p, f"code task {t['id']} does not discriminate"
        emit_pair(rows, f"code-{t['id']}", "code_spec",
                  canonical=template.format(t["func"], t["sig"], t["canonical"]),
                  perturbed=template.format(t["func"], t["sig"], t["perturbed"]),
                  hint=t["hint"], gold_canonical=expected_c, gold_perturbed=expected_p,
                  checker={"type": "python_tests", "func": t["func"], "args": t["tests"]},
                  suffix="\n\nReply with only the code, in a single ```python code block.",
                  meta={"ref_canonical": t["ref_canonical"], "ref_perturbed": t["ref_perturbed"]})


def build_dataset(seed):
    rng = random.Random(seed)
    rows = []
    for gen in (gen_puzzles, gen_arithmetic, gen_counting, gen_copy, gen_lists, gen_code):
        gen(rows, rng)
    return rows


DATASET = build_dataset(SEED)
DATASET_BY_ID = {r["id"]: r for r in DATASET}
CATEGORIES = list(dict.fromkeys(r["category"] for r in DATASET))
with open(DATA_PATH, "w", encoding="utf-8") as f:
    f.writelines(json.dumps(r, ensure_ascii=False) + "\n" for r in DATASET)
print(f"{len(DATASET)} rows -> {DATA_PATH}")
print(Counter(r["category"] for r in DATASET if r["condition"] == "canonical"))

### 1.3 Grader
`grade(row, response)` strips the thinking trace and labels the answer:

- `correct`: solved the task as asked
- `shortcut`: gave the answer to the familiar task
- `other_wrong`: wrong in some other way
- `ambiguous`: matches both the gold and the shortcut
- `unparseable`: no answer could be extracted

It also flags whether the answer, or the thinking trace, mentions the change or a "classic"/"famous" version.

In [ ]:
import math
import subprocess


def split_thinking(response):
    """Return (thinking, answer). Handles templates that pre-fill the opening <think> tag."""
    if "</think>" in response:
        thinking, answer = response.rsplit("</think>", 1)
        return thinking.replace("<think>", "").strip(), answer.strip()
    if "<think>" in response:  # generation stopped mid-thought
        return response.replace("<think>", "").strip(), ""
    return "", response.strip()


def final_answer(text):
    """The final answer: \\boxed{...}, else the last 'Answer: ...', else the last line."""
    boxed = re.findall(r"\\boxed\{((?:[^{}]|\{[^{}]*\})*)\}", text)
    if boxed:
        return boxed[-1].strip()
    clean = text.replace("*", "")
    found = re.findall(r"answer\s*[:：]\s*(.+)", clean, flags=re.I)
    if found:
        return found[-1].strip()
    lines = [line for line in clean.splitlines() if line.strip()]
    return lines[-1].strip() if lines else ""


NUMBER_WORDS = {w: i for i, w in enumerate(
    "zero one two three four five six seven eight nine ten eleven twelve thirteen fourteen "
    "fifteen sixteen seventeen eighteen nineteen twenty".split())}
NUMBER_WORDS.update({w: i for i, w in enumerate(
    "_ first second third fourth fifth sixth seventh eighth ninth tenth".split()) if i})


def parse_number(s):
    s = s.replace(",", "")
    s = re.sub(r"\\[dt]?frac\{([^{}]+)\}\{([^{}]+)\}", r"\1/\2", s)
    s = re.sub(r"base[\s-]*\d+|_\{?\d+\}?|[₀-₉]+", " ", s, flags=re.I)  # drop base markers like (base 9), _9
    if "=" in s:
        s = s.rsplit("=", 1)[1]
    m = re.search(r"(-?\d+(?:\.\d+)?)\s*/\s*(\d+(?:\.\d+)?)", s)
    if m and float(m[2]) != 0:
        return float(m[1]) / float(m[2])
    m = re.search(r"-?\d+(?:\.\d+)?", s)
    if m:
        value = float(m[0])
        return value / 100 if s[m.end():].lstrip().startswith("%") else value
    for word in re.findall(r"[a-z]+", s.lower()):
        if word in NUMBER_WORDS:
            return float(NUMBER_WORDS[word])
    return None


def close(a, b):
    return math.isclose(a, b, abs_tol=0.005)


def has_word(text, word, ignore_case=False):
    return re.search(rf"(?<!\w){re.escape(word)}(?!\w)", text, re.I if ignore_case else 0) is not None


LIST_ITEM_RE = re.compile(r"^\s*(?:\d+\s*[.):]|[-*•])\s+\S", re.M)


def normalize_copy(text):
    text = re.sub(r"^```\w*\s*|```$", "", text.strip()).strip()
    return re.sub(r"\s+", " ", text.strip("\"'`“”‘’ \n"))


def extract_code(text, func):
    blocks = re.findall(r"```(?:python|py)?\s*\n(.*?)```", text, flags=re.S)
    for block in blocks:
        if f"def {func}" in block:
            return block
    if blocks:
        return blocks[0]
    return text if f"def {func}" in text else None


# Model-written code runs in a separate, isolated Python process with a timeout.
HARNESS = r"""
import json, sys
spec = json.loads(sys.argv[1])
ns = {}
try:
    exec(sys.stdin.read(), ns)
except Exception:
    pass  # keep the function if it was defined before a crash (e.g. in example-usage code)
f = ns.get(spec["func"])
out = []
for args in spec["args"]:
    try:
        out.append(f(*args))
    except Exception as e:
        out.append("__error__: " + type(e).__name__)
print(json.dumps(out, default=str))
"""


def run_tests(code, func, args, timeout=10):
    try:
        proc = subprocess.run([sys.executable, "-I", "-c", HARNESS, json.dumps({"func": func, "args": args})],
                              input=code, capture_output=True, text=True, timeout=timeout)
        lines = proc.stdout.strip().splitlines()
        return json.loads(lines[-1]) if lines else None
    except (subprocess.TimeoutExpired, json.JSONDecodeError):
        return None


DISCLOSURE_RE = re.compile(
    r"\b(typo|misspel\w*|did you mean|(probably|likely) meant|assum\w*|presumably|"
    r"I think you mean\w*|correct(ed|ing)? (it|the|this)|usual(ly)? spel\w*|standard spelling)\b", re.I)
FAMILIAR_RE = re.compile(r"\b(classic|famous|well[- ]known|trick question|riddle)\b", re.I)


def _label(is_gold, is_shortcut):
    if is_gold and is_shortcut:
        return "ambiguous"
    return "correct" if is_gold else "shortcut" if is_shortcut else "other_wrong"


def grade(row, response):
    thinking, answer = split_thinking(response)
    result = {"label": "unparseable", "extracted": None,
              "discloses_change": bool(DISCLOSURE_RE.search(answer)),
              "mentions_familiar": bool(FAMILIAR_RE.search(answer)),
              "thinking_flags_change": bool(DISCLOSURE_RE.search(thinking)),
              "thinking_mentions_familiar": bool(FAMILIAR_RE.search(thinking)),
              "thinking_chars": len(thinking)}
    if not answer:
        return result
    kind, gold, shortcut = row["checker"]["type"], row["gold"], row["shortcut"]

    if kind == "number":
        value = parse_number(final_answer(answer))
        result["extracted"] = value
        if value is not None:
            result["label"] = _label(any(close(value, g) for g in gold),
                                     shortcut is not None and any(close(value, s) for s in shortcut))
    elif kind == "text":
        text = final_answer(answer)
        result["extracted"] = text
        result["label"] = _label(any(has_word(text, g, True) for g in gold),
                                 shortcut is not None and any(has_word(text, s, True) for s in shortcut))
    elif kind == "list_count":
        n = len(LIST_ITEM_RE.findall(answer))
        result["extracted"] = n
        if n:
            result["label"] = _label(n == gold, shortcut is not None and n in row["checker"]["shortcut_counts"])
    elif kind == "exact_copy":
        text = normalize_copy(answer)
        result["extracted"] = text
        if text == gold["text"]:
            result["label"] = "correct"
        elif shortcut is not None and text == shortcut["text"]:
            result["label"] = "shortcut"
        else:  # not an exact copy: did the changed word survive, or was it reverted?
            has_gold = has_word(text, gold["token"])
            has_short = shortcut is not None and has_word(text, shortcut["token"])
            result["label"] = _label(has_gold and not has_short, has_short and not has_gold)
    elif kind == "python_tests":
        code = extract_code(answer, row["checker"]["func"])
        if code is not None:
            outputs = run_tests(code, row["checker"]["func"], row["checker"]["args"])
            result["extracted"] = outputs
            result["label"] = _label(outputs == gold, shortcut is not None and outputs == shortcut)
    else:
        raise ValueError(f"unknown checker type: {kind}")
    return result

### 1.4 Self-test
Check the grader before spending GPU time. For every row, a synthetic perfect answer must be graded `correct`. On non-canonical rows, a synthetic familiar answer must be graded `shortcut`. The test also covers answer formats the model is likely to produce.

In [ ]:
def synthetic_response(row, value, canonical_ref):
    kind = row["checker"]["type"]
    if kind in ("number", "text"):
        return f"Let me think about this.\nAnswer: {value[0]}"
    if kind == "list_count":
        return "\n".join(f"{i}. item {i}" for i in range(1, value + 1))
    if kind == "exact_copy":
        return value["text"]
    return f"```python\n{row['meta']['ref_canonical' if canonical_ref else 'ref_perturbed']}```"


problems = []
for row in DATASET:
    canonical = row["condition"] == "canonical"
    if (got := grade(row, synthetic_response(row, row["gold"], canonical))["label"]) != "correct":
        problems.append((row["id"], "gold", got))
    if not canonical:
        if row["gold"] == row["shortcut"]:
            problems.append((row["id"], "gold == shortcut"))
        value = 10 if row["checker"]["type"] == "list_count" else row["shortcut"]
        if (got := grade(row, synthetic_response(row, value, True))["label"]) != "shortcut":
            problems.append((row["id"], "shortcut", got))

for text, want in [("**Answer:** $0.10", 0.10), ("so it is \\boxed{\\frac{1}{2}}", 0.5),
                   ("Answer: 106 (base 9)", 106), ("Answer: In base 9, 47 + 38 = 106", 106),
                   ("Answer: 106_9", 106), ("Answer: 66.7%", 0.667), ("Answer: third place", 3),
                   ("Answer: 3rd", 3), ("Answer: zero", 0), ("It costs 10 cents.\n\nAnswer: 10 cents", 10),
                   ("Answer:\n47", 47)]:
    got = parse_number(final_answer(text))
    if got is None or not close(got, want):
        problems.append(("parse", text, got))

think_row = {"checker": {"type": "number"}, "gold": [10], "shortcut": [5]}
for text, want in [("<think>The classic answer is 5, but the bat costs $1.00.</think>\nAnswer: 10", "correct"),
                   ("The classic answer is 5.</think>\n\nAnswer: 5", "shortcut"),
                   ("<think>Hmm, let me think about this for a very long", "unparseable")]:
    if (got := grade(think_row, text)["label"]) != want:
        problems.append(("thinking", text, got))

for p in problems:
    print("FAIL", p)
print(f"{len(DATASET)} rows checked, {len(problems)} problems")
assert not problems

In [ ]:
import pandas as pd

df_data = pd.DataFrame(DATASET)
display(pd.crosstab(df_data.category, df_data.condition))
for category, group in df_data.groupby("category", sort=False):
    pair = group[group.pair_id == group.pair_id.iloc[0]]
    print("=" * 100, f"\n{category}  ({pair.pair_id.iloc[0]})")
    for _, r in pair.iterrows():
        print(f"\n--- {r.condition}   gold={r.gold!r}   shortcut={r.shortcut!r}\n{r.prompt}")

## 2 · Model

Qwen3.5-4B is multimodal, but every prompt here is text-only. Thinking is on by default; the chat template's `enable_thinking` flag switches it off for the `no_think` mode.

In [ ]:
import torch

assert torch.cuda.is_available(), "Switch to a GPU runtime first."
DTYPE = "bfloat16" if torch.cuda.get_device_capability()[0] >= 8 else "float16"   # T4 has no bf16
print(torch.cuda.get_device_name(), DTYPE)
SPECIAL_TOKEN_RE = re.compile(r"<\|[^|>]*\|>")

if ENGINE == "vllm":
    from vllm import LLM, SamplingParams

    llm = LLM(model=MODEL_ID, dtype=DTYPE, seed=SEED, gpu_memory_utilization=0.85,
              max_model_len=max(MAX_NEW_TOKENS.values()) + 1024,
              limit_mm_per_prompt={"image": 0, "video": 0})   # text-only prompts

    def generate(prompts, mode):
        params = SamplingParams(n=N_SAMPLES, max_tokens=MAX_NEW_TOKENS[mode], seed=SEED,
                                skip_special_tokens=False, **SAMPLING[mode])
        outs = llm.chat([[{"role": "user", "content": p}] for p in prompts], sampling_params=params,
                        use_tqdm=False, chat_template_kwargs={"enable_thinking": mode == "think"})
        return [[(SPECIAL_TOKEN_RE.sub("", c.text), c.finish_reason == "length", len(c.token_ids))
                 for c in o.outputs] for o in outs]

    CHUNK = 128
else:
    from transformers import AutoModelForMultimodalLM, AutoProcessor

    processor = AutoProcessor.from_pretrained(MODEL_ID)
    tokenizer = processor.tokenizer
    tokenizer.padding_side = "left"
    model = AutoModelForMultimodalLM.from_pretrained(MODEL_ID, dtype=getattr(torch, DTYPE),
                                                     device_map="auto").eval()
    STOP_IDS = {tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|im_end|>")}

    def generate(prompts, mode):
        texts = [processor.apply_chat_template([{"role": "user", "content": [{"type": "text", "text": p}]}],
                                               tokenize=False, add_generation_prompt=True,
                                               enable_thinking=mode == "think") for p in prompts]
        inputs = tokenizer(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        with torch.no_grad():
            seqs = model.generate(**inputs, do_sample=True, max_new_tokens=MAX_NEW_TOKENS[mode],
                                  num_return_sequences=N_SAMPLES, **SAMPLING[mode])
        new = seqs[:, inputs["input_ids"].shape[1]:].tolist()
        results = []
        for i in range(len(prompts)):
            group = []
            for ids in new[i * N_SAMPLES:(i + 1) * N_SAMPLES]:
                text = SPECIAL_TOKEN_RE.sub("", tokenizer.decode(ids, skip_special_tokens=False))
                n_tokens = sum(t != tokenizer.pad_token_id for t in ids)
                group.append((text, not STOP_IDS.intersection(ids), n_tokens))
            results.append(group)
        return results

    torch.manual_seed(SEED)
    CHUNK = HF_BATCH_SIZE

### 2.1 Smoke test
Before the full run, check one perturbed prompt in each mode.

In [ ]:
probe = DATASET_BY_ID["puzzle-bat_ball-perturbed"]
for mode in MODES:
    text, truncated, n = generate([probe["prompt"]], mode)[0][0]
    print(f"===== {mode}  ({n} tokens, truncated={truncated})\n{text[-1500:]}\n")
    print(grade(probe, text))

### 2.2 Generate
Responses are appended to `results/generations_<mode>.jsonl` and synced to the bucket after every chunk. Rows that are already done are skipped. If you change `N_SAMPLES`, delete the old generation files first.

In [ ]:
import time


def select_rows(pairs_per_category):
    if not pairs_per_category:
        return DATASET
    keep = set()
    for category in CATEGORIES:
        pair_ids = list(dict.fromkeys(r["pair_id"] for r in DATASET if r["category"] == category))
        keep.update(pair_ids[:pairs_per_category])
    return [r for r in DATASET if r["pair_id"] in keep]


def read_jsonl(path):
    rows = []
    if path.exists():
        for line in open(path, encoding="utf-8"):
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                pass  # a line cut off by a disconnect; that row is simply regenerated
    return rows


for mode in MODES:
    path = OUT / "results" / f"generations_{mode}.jsonl"
    done = {g["id"] for g in read_jsonl(path)}
    todo = [r for r in select_rows(PAIRS_PER_CATEGORY[mode]) if r["id"] not in done]
    print(f"{mode}: {len(done)} rows done, {len(todo)} to go")
    start = time.time()
    for k in range(0, len(todo), CHUNK):
        batch = todo[k:k + CHUNK]
        outputs = generate([r["prompt"] for r in batch], mode)
        lines = [json.dumps({"id": r["id"], "mode": mode, "sample": s, "response": text,
                             "truncated": truncated, "n_tokens": n_tokens}, ensure_ascii=False) + "\n"
                 for r, samples in zip(batch, outputs)
                 for s, (text, truncated, n_tokens) in enumerate(samples)]
        with open(path, "a", encoding="utf-8") as f:
            f.write("".join(lines))
        sync_up()
        print(f"  {k + len(batch)}/{len(todo)}   {time.time() - start:.0f}s")

## 3 · Grading

In [ ]:
records = []
for mode in MODES:
    for g in read_jsonl(OUT / "results" / f"generations_{mode}.jsonl"):
        row = DATASET_BY_ID[g["id"]]
        response = g["response"]
        if mode == "think" and "</think>" not in response:
            response = "<think>" + response   # thinking never finished: no answer to grade
        records.append({"id": row["id"], "pair_id": row["pair_id"], "category": row["category"],
                        "condition": row["condition"], "mode": mode, "sample": g["sample"],
                        "truncated": g["truncated"], "n_tokens": g["n_tokens"],
                        **grade(row, response), "response": g["response"]})

graded = pd.DataFrame(records)
graded.to_csv(OUT / "results" / "graded.csv", index=False)
sync_up()
pd.crosstab([graded["mode"], graded.condition], graded.label)

## 4 · Analysis

How to read the summary (one row per mode × category, plus `ALL`):

| column | meaning |
|---|---|
| `acc_<condition>` | share of answers that are correct under each condition |
| `shortcut_rate` | share of `perturbed` answers that equal the familiar answer |
| `subst_share_of_errors` | among wrong `perturbed` answers, the share that is the familiar answer, with a 95% bootstrap CI over pairs |
| `recovery_nudge`, `recovery_explicit` | accuracy gain over `perturbed`: the model had the skill but did not use it |
| `silent_rate` | among shortcut answers (`perturbed` + `nudge`), the share that never mention the change |
| `noticed_in_thinking` | think mode only: among shortcut answers, the share whose reasoning mentioned the change or the classic version |

**What supports the hypothesis:**
- high `acc_canonical`
- a sharp drop at `perturbed`
- a high `subst_share_of_errors`
- large recovery
- mostly silent substitutions

In [ ]:
import numpy as np

LABELS = ["correct", "shortcut", "other_wrong", "ambiguous", "unparseable"]
CAT_NAMES = {"modified_puzzle": "Modified puzzles", "counterfactual_arithmetic": "Base-8/9 arithmetic",
             "misspelled_counting": "Misspelled counting", "verbatim_copy": "Verbatim copy",
             "list_length": "List length", "code_spec": "Code spec"}
MODE_NAMES = {"no_think": "Thinking off", "think": "Thinking on"}
rng = np.random.default_rng(SEED)


def boot_ci(num, den, n=2000):
    """95% CI for sum(num) / sum(den), resampling pairs with replacement."""
    num, den = np.asarray(num, float), np.asarray(den, float)
    if den.sum() == 0:
        return np.nan, np.nan
    idx = rng.integers(0, len(num), (n, len(num)))
    d = den[idx].sum(1)
    ratios = num[idx].sum(1) / np.where(d == 0, np.nan, d)
    return tuple(np.nanpercentile(ratios, [2.5, 97.5]))


def summarize(d):
    acc = {c: (d[d.condition == c].label == "correct").mean() for c in CONDITIONS}
    p = d[d.condition == "perturbed"]
    per_pair = p.groupby("pair_id").label.agg(shortcut=lambda s: (s == "shortcut").sum(),
                                              errors=lambda s: s.isin(["shortcut", "other_wrong"]).sum())
    lo, hi = boot_ci(per_pair.shortcut, per_pair.errors)
    subs = d[d.condition.isin(["perturbed", "nudge"]) & (d.label == "shortcut")]
    think_subs = p[(p.label == "shortcut") & (p["mode"] == "think")]
    return {
        "pairs": d.pair_id.nunique(),
        **{f"acc_{c}": v for c, v in acc.items()},
        "shortcut_rate": (p.label == "shortcut").mean(),
        "subst_share_of_errors": per_pair.shortcut.sum() / max(per_pair.errors.sum(), 1),
        "subst_ci": "" if np.isnan(lo) else f"[{lo:.0%}, {hi:.0%}]",
        "recovery_nudge": acc["nudge"] - acc["perturbed"],
        "recovery_explicit": acc["explicit"] - acc["perturbed"],
        "silent_rate": (~subs.discloses_change).mean() if len(subs) else np.nan,
        "noticed_in_thinking": ((think_subs.thinking_flags_change | think_subs.thinking_mentions_familiar).mean()
                                if len(think_subs) else np.nan),
        "truncated": d.truncated.mean(),
    }


summary = pd.DataFrame([{"mode": m, "category": c, **summarize(d)}
                        for m, dm in graded.groupby("mode", sort=False)
                        for c, d in [*dm.groupby("category", sort=False), ("ALL", dm)]])
summary.to_csv(OUT / "results" / "summary.csv", index=False)
pct_cols = [c for c in summary.columns if c.startswith(("acc_", "recovery_")) or
            c in ("shortcut_rate", "subst_share_of_errors", "silent_rate", "noticed_in_thinking", "truncated")]
display(summary.style.format({c: "{:.0%}" for c in pct_cols}, na_rep="–"))

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

SURFACE, INK, INK2, MUTED, GRID, BASELINE = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": BASELINE, "axes.labelcolor": INK2, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelcolor": INK2, "ytick.labelcolor": INK2,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
    "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold", "legend.frameon": False,
})
OUTCOME_COLORS = {"correct": "#2a78d6", "shortcut": "#eb6834", "other wrong": "#1baf7a", "no answer": BASELINE}
MODE_STYLES = {"no_think": ("#2a78d6", "-"), "think": ("#eb6834", "--")}
modes_run = [m for m in MODES if m in set(graded["mode"])]

# Figure 1: what the model answered on the perturbed task.
pert = graded[graded.condition == "perturbed"]
mix = (pd.crosstab([pert["mode"], pert.category], pert.label, normalize="index")
         .reindex(columns=LABELS, fill_value=0))
mix["other wrong"] = mix.pop("other_wrong")
mix["no answer"] = mix.pop("ambiguous") + mix.pop("unparseable")

fig, axes = plt.subplots(1, len(modes_run), figsize=(6.2 * len(modes_run), 4.4), sharey=True, squeeze=False)
for ax, mode in zip(axes[0], modes_run):
    d = mix.loc[mode].reindex(CATEGORIES[::-1]).fillna(0)
    names = [CAT_NAMES[c] for c in d.index]
    left = np.zeros(len(d))
    for outcome, color in OUTCOME_COLORS.items():
        vals = d[outcome].to_numpy()
        ax.barh(names, vals, left=left, height=0.62, color=color, edgecolor=SURFACE, linewidth=2, label=outcome)
        if outcome == "shortcut":
            for y, (l, v) in enumerate(zip(left, vals)):
                if v >= 0.08:
                    ax.text(l + v / 2, y, f"{v:.0%}", ha="center", va="center", fontsize=9, color=INK)
        left += vals
    ax.set_xlim(0, 1)
    ax.xaxis.set_major_formatter(PercentFormatter(1))
    ax.grid(axis="y", visible=False)
    ax.tick_params(axis="y", length=0)
    ax.set_title(MODE_NAMES[mode], loc="left")
handles, labels = axes[0][0].get_legend_handles_labels()
fig.legend(handles, labels, ncol=4, loc="upper left", bbox_to_anchor=(0.01, 0.92))
fig.suptitle("Answers to the perturbed task: how often the model solved the familiar one instead",
             x=0.01, ha="left", fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.84))
fig.savefig(OUT / "figures" / "outcome_mix_perturbed.png", dpi=200)
plt.show()

# Figure 2: accuracy across the four conditions, one panel per category.
acc = graded.assign(ok=graded.label.eq("correct")).pivot_table(
    index=["mode", "category"], columns="condition", values="ok", aggfunc="mean").reindex(columns=CONDITIONS)
fig, axes = plt.subplots(2, 3, figsize=(12, 6.6), sharex=True, sharey=True)
for i, (ax, category) in enumerate(zip(axes.flat, CATEGORIES)):
    for mode in modes_run:
        if (mode, category) not in acc.index:
            continue
        color, style = MODE_STYLES[mode]
        y = acc.loc[(mode, category)].to_numpy()
        ax.plot(range(len(CONDITIONS)), y, style, color=color, lw=2, marker="o", ms=7,
                markeredgecolor=SURFACE, markeredgewidth=1.5, label=MODE_NAMES[mode])
        if i == 0:  # direct labels once; the legend covers the rest
            ax.annotate(MODE_NAMES[mode], (len(CONDITIONS) - 1, y[-1]), xytext=(8, 0),
                        textcoords="offset points", va="center", fontsize=9, color=INK2)
    ax.set_title(CAT_NAMES[category], loc="left")
    ax.set_ylim(0, 1.05)
    ax.set_xlim(-0.25, len(CONDITIONS) - 0.25 + 0.9)
    ax.yaxis.set_major_formatter(PercentFormatter(1))
    ax.set_xticks(range(len(CONDITIONS)), CONDITIONS)
    ax.grid(axis="x", visible=False)
handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, ncol=2, loc="upper left", bbox_to_anchor=(0.01, 0.93))
fig.suptitle("Accuracy by condition: familiar task, changed task, and the changed task with a warning or hint",
             x=0.01, ha="left", fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.87))
fig.savefig(OUT / "figures" / "accuracy_by_condition.png", dpi=200)
plt.show()
sync_up()

### 4.1 Examples
Read real substitutions: these examples are the qualitative evidence for the write-up.

In [ ]:
def show_examples(label="shortcut", condition="perturbed", mode="no_think", per_category=2):
    sub = graded[(graded.label == label) & (graded.condition == condition) & (graded["mode"] == mode)]
    for category, group in sub.groupby("category", sort=False):
        for _, r in group.head(per_category).iterrows():
            row = DATASET_BY_ID[r.id]
            thinking, answer = split_thinking(r.response)
            print("=" * 100)
            print(f"[{CAT_NAMES[category]}] {r.id}\n  gold={row['gold']!r}\n  shortcut={row['shortcut']!r}\n"
                  f"  extracted={r.extracted!r}   disclosed={r.discloses_change}")
            print("PROMPT:\n" + row["prompt"])
            if thinking:
                print("THINKING (end):\n..." + thinking[-800:])
            print("ANSWER:\n" + answer[-800:])


show_examples(mode=modes_run[0])

## 5 · Publish

Makes the bucket public. Run this once the results are final. Then upload the README (with the answers to Q1 and Q3) and this notebook, either through the bucket's web page or with `hf buckets cp <file> hf://buckets/<your-bucket>/` from your PC.

In [ ]:
from huggingface_hub import update_bucket_settings

sync_up()
update_bucket_settings(HF_BUCKET, private=False)
print("Public bucket:", f"https://huggingface.co/buckets/{HF_BUCKET}")